## 2022-2025 Expenditure Features

In [43]:
# Theil-Sen Regression: predict income growth from six tourism expenditure features

import pandas as pd
from sklearn.linear_model import TheilSenRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = "data 2022-2025.csv"
INCOME_COLUMN = "Income mean (RM)"
TARGET_COLUMN = "Income growth (%)"
HOLDOUT_YEAR = 2025
EXPENDITURE_COLUMNS = [
    "Shopping expenditure",
    "Purchase of automotive fuel expenditure",
    "Transportation expenditure",
    "Food & beverage expenditure",
    "Accomodation expenditure",
    "Entrance ticket expenditure",
]

data = pd.read_csv(DATA_PATH).sort_values(["State", "Year"])

# Calculate year-over-year income growth separately for each state.
data[TARGET_COLUMN] = data.groupby("State")[INCOME_COLUMN].pct_change() * 100
model_data = data.dropna(subset=[TARGET_COLUMN] + EXPENDITURE_COLUMNS).copy()

train = model_data[model_data["Year"] < HOLDOUT_YEAR].copy()
test = model_data[model_data["Year"] == HOLDOUT_YEAR].copy()

model = TheilSenRegressor(random_state=42, max_subpopulation=10_000)
model.fit(train[EXPENDITURE_COLUMNS], train[TARGET_COLUMN])
test["Predicted income growth (%)"] = model.predict(test[EXPENDITURE_COLUMNS])

prediction_results = test[
    ["State", "Year"] + EXPENDITURE_COLUMNS + [TARGET_COLUMN, "Predicted income growth (%)"]
].sort_values("State")
prediction_results["Absolute error (percentage points)"] = (
    prediction_results[TARGET_COLUMN]
    - prediction_results["Predicted income growth (%)"]
).abs()

print(f"Training rows: {len(train)}")
print(f"Prediction rows: {len(test)}")
print(f"Mean absolute error: {mean_absolute_error(test[TARGET_COLUMN], test['Predicted income growth (%)']):.2f} percentage points")
print(f"Root mean squared error: {mean_squared_error(test[TARGET_COLUMN], test['Predicted income growth (%)']) ** 0.5:.2f} percentage points")
print(f"R²: {r2_score(test[TARGET_COLUMN], test['Predicted income growth (%)']):.3f}")
print("Theil-Sen coefficients:")
for feature, coefficient in zip(EXPENDITURE_COLUMNS, model.coef_):
    print(f"  {feature}: {coefficient:.8f}")

prediction_results

Training rows: 26
Prediction rows: 13
Mean absolute error: 0.75 percentage points
Root mean squared error: 0.98 percentage points
R²: -0.231
Theil-Sen coefficients:
  Shopping expenditure: 0.00000100
  Purchase of automotive fuel expenditure: 0.00000214
  Transportation expenditure: -0.00000146
  Food & beverage expenditure: -0.00000371
  Accomodation expenditure: 0.00000357
  Entrance ticket expenditure: -0.00000583


,State,Year,Shopping expenditure,Purchase of automotive fuel expenditure,Transportation expenditure,Food & beverage expenditure,Accomodation expenditure,Entrance ticket expenditure,Income growth (%),Predicted income growth (%),Absolute error (percentage points)
3,Johor,2025,2746983,1210563,692665,1333180,1131267,316602,5.098060,4.403499,0.694561
7,Kedah,2025,1576858,804194,519526,1040119,587964,223964,2.097359,2.304006,0.206647
11,Kelantan,2025,1567481,934127,362241,925230,486942,104703,3.608737,3.562482,0.046255
15,Melaka,2025,3125361,1192861,484995,1460170,954518,364060,3.620769,3.668553,0.047784
19,Negeri Sembilan,2025,2166545,936212,417566,1151803,567601,427983,3.538672,1.649032,1.889641
23,Pahang,2025,2965383,1404852,608291,1444247,1312310,498537,2.382130,4.334198,1.952068
27,Penang,2025,3423333,1019165,512166,1378621,1079517,368571,4.835009,4.279214,0.555795
31,Perak,2025,2250351,1390860,543730,1637041,795667,228115,3.191317,2.697697,0.493620
35,Perlis,2025,327519,213863,101611,246445,118121,35913,2.768512,2.766409,0.002104
39,Sabah,2025,3589154,1250952,836013,1459229,1073566,508780,2.516159,3.330048,0.813889


## Theil-Sen Regression: predict income growth from tourism receipts (2018-2024)
# COVID flag added; 2020 and 2021 dropped from modelling.

In [42]:
# Theil-Sen Regression: predict income growth from tourism receipts
# COVID flag added; 2020 and 2021 dropped from modelling.

import pandas as pd
from sklearn.linear_model import TheilSenRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = "data.csv"
INCOME_COLUMN = "Income mean (RM)"
TARGET_COLUMN = "Income growth (%)"
RECEIPTS_COLUMN = "Tourism receipts (RM)"
COVID_YEARS = [2020, 2021]

data = pd.read_csv(DATA_PATH)
data.columns = data.columns.str.strip()
for col in ["Year", INCOME_COLUMN, RECEIPTS_COLUMN]:
    data[col] = pd.to_numeric(data[col], errors="coerce")
data = data.sort_values(["State", "Year"]).reset_index(drop=True)

# Track real (non-interpolated) income values
data["Income observed"] = data[INCOME_COLUMN].notna()

# Interpolate only between known values (no extrapolation into 2025)
data[INCOME_COLUMN] = data.groupby("State")[INCOME_COLUMN].transform(
    lambda x: x.interpolate(method="linear", limit_area="inside")
)

# Year-over-year growth, computed BEFORE dropping COVID years
data[TARGET_COLUMN] = (
    data.groupby("State")[INCOME_COLUMN].pct_change(fill_method=None) * 100
)

# COVID indicator
data["COVID"] = data["Year"].isin(COVID_YEARS).astype(int)

model_data = data.dropna(subset=[TARGET_COLUMN, RECEIPTS_COLUMN]).copy()

# Drop COVID years from modelling
model_data = model_data[model_data["COVID"] == 0].copy()

# Holdout = latest year with observed income
HOLDOUT_YEAR = int(model_data.loc[model_data["Income observed"], "Year"].max())
train = model_data[model_data["Year"] < HOLDOUT_YEAR].copy()
test = model_data[
    (model_data["Year"] == HOLDOUT_YEAR) & model_data["Income observed"]
].copy()

# The COVID flag is constant (0) once COVID years are dropped, so it can't
# carry signal; only use it as a feature if it varies in the training set.
FEATURES = [RECEIPTS_COLUMN]
if train["COVID"].nunique() > 1:
    FEATURES.append("COVID")

print("Train years:", sorted(train["Year"].unique()), "| rows:", len(train))
print("Holdout:", HOLDOUT_YEAR, "| rows:", len(test))
print("Features:", FEATURES)

if train.empty or test.empty:
    raise ValueError("Empty train or test set.")

model = TheilSenRegressor(random_state=42, max_subpopulation=10_000)
model.fit(train[FEATURES], train[TARGET_COLUMN])
test["Predicted income growth (%)"] = model.predict(test[FEATURES])

y, p = test[TARGET_COLUMN], test["Predicted income growth (%)"]
baseline = train[TARGET_COLUMN].mean()

print("\nModel Performance")
print("-------------------------")
print(f"MAE:  {mean_absolute_error(y, p):.2f} pp")
print(f"RMSE: {mean_squared_error(y, p) ** 0.5:.2f} pp")
print(f"R²:   {r2_score(y, p):.3f}")
print(f"Baseline MAE (predict train mean {baseline:.2f}%): "
      f"{mean_absolute_error(y, [baseline] * len(y)):.2f} pp")
print("\nCoefficients:", dict(zip(FEATURES, model.coef_)))

prediction_results = test[
    ["State", "Year", RECEIPTS_COLUMN, TARGET_COLUMN, "Predicted income growth (%)"]
].sort_values("State")
prediction_results["Absolute error (pp)"] = (y - p).abs()
print(prediction_results.to_string(index=False))

Train years: [np.int64(2022), np.int64(2023)] | rows: 26
Holdout: 2024 | rows: 13
Features: ['Tourism receipts (RM)']

Model Performance
-------------------------
MAE:  1.25 pp
RMSE: 1.52 pp
R²:   -1.539
Baseline MAE (predict train mean 5.57%): 2.08 pp

Coefficients: {'Tourism receipts (RM)': np.float64(-2.979795907255411e-10)}
          State  Year  Tourism receipts (RM)  Income growth (%)  Predicted income growth (%)  Absolute error (pp)
          Johor  2024             7814000000           5.371924                     3.572583             1.799341
          Kedah  2024             4961000000           2.142290                     4.422718             2.280428
       Kelantan  2024             4541000000           3.743842                     4.547870             0.804027
         Melaka  2024             7933000000           3.756794                     3.537123             0.219671
Negeri Sembilan  2024             5869000000           3.668488                     4.152153        

## Theil-Sen Regression: predict income growth from expenditures with theil-sen regression. use data 2022-2025.csv

In [29]:
import pandas as pd
import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import TheilSenRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("data 2022-2025.csv")

df = df.sort_values(["State", "Year"]).copy()


# ============================================================
# 2. TOTAL TOURISM EXPENDITURE
# ============================================================

expenditure_cols = [
    "Shopping expenditure",
    "Purchase of automotive fuel expenditure",
    "Transportation expenditure",
    "Food & beverage expenditure",
    "Accomodation expenditure",
    "Entrance ticket expenditure",
    "Other expenditure",
    "Visited household expediture"
]

df["Total tourism expenditure"] = df[expenditure_cols].sum(axis=1)


# ============================================================
# 3. TARGET: REAL MEDIAN INCOME GROWTH
# ============================================================

df["Income growth (%)"] = (
    df.groupby("State")["Income median (RM)"]
      .pct_change() * 100
)

df["Real income growth (%)"] = (
    (
        (1 + df["Income growth (%)"] / 100)
        / (1 + df["Inflation (%)"] / 100)
    ) - 1
) * 100


# ============================================================
# 4. TOURISM INTENSITY FEATURES
# ============================================================

df["Tourism expenditure per household"] = (
    df["Total tourism expenditure"]
    / df["Household"]
)

df["Tourism receipts per household"] = (
    df["Tourism receipts (RM)"]
    / df["Household"]
)

df["Tourism expenditure per visitor"] = (
    df["Total tourism expenditure"]
    / df["Domestic visitor"]
)


# ============================================================
# 5. CREATE LAGGED FEATURES
#
# Previous year's tourism conditions
# -> current year's real income growth
# ============================================================

feature_cols = [
    "Tourism expenditure per household",
    "Tourism receipts per household",
    "Domestic visitor",
    "Tourism expenditure per visitor",
    "Receipts per visitor (RM)",
    "Average length of stay (day)",
    "Income median (RM)"
]

for col in feature_cols:
    df[f"{col} lag1"] = (
        df.groupby("State")[col].shift(1)
    )


# ============================================================
# 6. FEATURES
# ============================================================

X_columns = [
    "Tourism expenditure per household lag1",
    "Tourism receipts per household lag1",
    "Domestic visitor lag1",
    "Tourism expenditure per visitor lag1",
    "Receipts per visitor (RM) lag1",
    "Average length of stay (day) lag1",
    "Income median (RM) lag1"
]

target = "Real income growth (%)"


# ============================================================
# 7. REMOVE MISSING VALUES
# ============================================================

model_df = df[
    X_columns +
    [target, "Year", "State"]
].dropna().copy()


# ============================================================
# 8. TIME-BASED SPLIT
# ============================================================

train = model_df[model_df["Year"] < 2025]
test = model_df[model_df["Year"] == 2025]

X_train = train[X_columns]
y_train = train[target]

X_test = test[X_columns]
y_test = test[target]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


# ============================================================
# 9. THEIL-SEN MODEL
# ============================================================

model = make_pipeline(
    StandardScaler(),

    TheilSenRegressor(
        random_state=42,
        max_subpopulation=10_000
    )
)


# ============================================================
# 10. TRAIN
# ============================================================

model.fit(X_train, y_train)


# ============================================================
# 11. PREDICT
# ============================================================

y_pred = model.predict(X_test)


# ============================================================
# 12. EVALUATION
# ============================================================

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("\nModel Performance")
print("-------------------------")
print(f"MAE :  {mae:.3f}%")
print(f"RMSE:  {rmse:.3f}%")
print(f"R²  :  {r2:.3f}")


# ============================================================
# 13. ACTUAL VS PREDICTED
# ============================================================

results = test[["State", "Year"]].copy()

results["Actual real income growth (%)"] = y_test.values
results["Predicted real income growth (%)"] = y_pred

results["Error (%)"] = (
    results["Predicted real income growth (%)"]
    - results["Actual real income growth (%)"]
)

results = results.sort_values("State")

print("\nPredictions")
print(results.to_string(index=False))


# ============================================================
# 14. COEFFICIENTS
# ============================================================

scaler = model.named_steps["standardscaler"]
theil_sen = model.named_steps["theilsenregressor"]

coefficients = theil_sen.coef_ / scaler.scale_

coefficient_df = pd.DataFrame({
    "Feature": X_columns,
    "Coefficient": coefficients
})

coefficient_df["Absolute coefficient"] = (
    coefficient_df["Coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    "Absolute coefficient",
    ascending=False
)

print("\nFeature Coefficients")
print(coefficient_df.to_string(index=False))

Training samples: 26
Testing samples: 13

Model Performance
-------------------------
MAE :  1.110%
RMSE:  1.498%
R²  :  -0.074

Predictions
          State  Year  Actual real income growth (%)  Predicted real income growth (%)  Error (%)
          Johor  2025                       3.374533                          4.369310   0.994777
          Kedah  2025                       4.253847                          1.874050  -2.379797
       Kelantan  2025                       5.385017                          2.256505  -3.128512
         Melaka  2025                       3.400559                          3.536584   0.136024
Negeri Sembilan  2025                       1.558000                          1.769968   0.211968
         Pahang  2025                       1.028912                          1.170656   0.141745
         Penang  2025                       4.779332                          4.505345  -0.273987
          Perak  2025                       1.208733                       

In [24]:
import pandas as pd
import numpy as np

# Import 2018-2025 data
data = pd.read_csv("data.csv")
# data.head()
data["Year"].unique()

array([2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])